In [0]:
import csv, re
from pyspark.sql import functions as F

landing = "/Volumes/cloud_pricing/raw/landing/aws_ec2"
raw_table = "cloud_pricing.bronze.aws_ec2_prices_raw"
audit_table = "cloud_pricing.bronze.aws_load_audit"
# first 5 lines of the aws csv are file info, 6th is the header
meta_rows = ["FormatVersion", "Disclaimer", "Publication Date", "Version", "OfferCode", "SKU"]

spark.sql(f"create table if not exists {audit_table} (version string, region string, rows bigint, keys bigint, status string, checked_at timestamp)")

In [0]:
done = [(r.version, r.region) for r in spark.sql(f"select distinct version, region from {audit_table}").collect()]

todo = []
for v in dbutils.fs.ls(landing):
    version = v.name.strip("/").split("=")[1]
    for r in dbutils.fs.ls(v.path):
        region = r.name.strip("/")
        if (version, region) not in done:
            todo.append((version, region, r.path + "index.csv"))
print(todo)

In [0]:
def header_columns(path):
    line = spark.read.text(path).filter(F.col("value").startswith('"SKU"')).first()["value"]
    names = next(csv.reader([line]))
    # delta doesn't allow spaces, / etc in column names
    return [re.sub(r"[^0-9a-zA-Z]+", "_", n).strip("_").lower() for n in names]


for version, region, path in todo:
    cols = header_columns(path)
    schema = ", ".join(f"`{c}` string" for c in cols)

    # schema given explicitly, otherwise spark takes column count from line 1 which only has 2 fields
    df = (spark.read.schema(schema).option("header", False).csv(path)
          .filter(~F.col("sku").isin(meta_rows))
          .withColumn("version", F.lit(version))
          .withColumn("version_ts", F.to_timestamp(F.lit(version), "yyyyMMddHHmmss"))
          .withColumn("region", F.lit(region))
          .withColumn("source_file", F.col("_metadata.file_path"))
          .withColumn("ingested_at", F.current_timestamp()))

    s = df.agg(F.count("*").alias("rows"), F.countDistinct("ratecode").alias("keys"),
               F.sum(F.col("ratecode").isNull().cast("int")).alias("null_keys")).first()
    status = "loaded" if s["rows"] > 0 and s["null_keys"] == 0 and s["keys"] == s["rows"] else "incomplete"

    if status == "loaded":
        # aws adds columns now and then, mergeSchema so a new version doesn't break the write
        (df.write.mode("overwrite")
           .option("replaceWhere", f"version = '{version}' and region = '{region}'")
           .option("mergeSchema", "true")
           .saveAsTable(raw_table))

    spark.createDataFrame([(version, region, s["rows"], s["keys"], status)], "version string, region string, rows bigint, keys bigint, status string") \
        .withColumn("checked_at", F.current_timestamp()).write.mode("append").saveAsTable(audit_table)
    print(version, region, status, s["rows"])